## AuxTel Tracking tests

Craig Lage 22-Mar-21

Mike Warner requested tracking tests in the day with closed dome.\
Near the SCP with very slow tracking speeds

In [1]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from lsst.ts.observatory.control.utils import RotType
from astropy.time import Time, TimeDelta
from astropy.coordinates import AltAz, ICRS, EarthLocation, Angle, FK5
import astropy.units as u

In [2]:
# Set Cerro Pachon location
location = EarthLocation.from_geodetic(lon=-70.747698*u.deg,
                                       lat=-30.244728*u.deg,
                                       height=2663.0*u.m)

In [3]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [4]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [5]:
#get classes and start them
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.01 sec
Read 1 history items for RemoteEvent(ATSpectrograph, 0, appliedSettingsMatchStart)
Read 100 history items for RemoteEvent(ATSpectrograph, 0, heartbeat)
Read 1 history items for RemoteEvent(ATSpectrograph, 0, reportedDisperserPosition)
Read 1 history items for RemoteEvent(ATSpectrograph, 0, reportedFilterPosition)
Read 1 history items for RemoteEvent(ATSpectrograph, 0, reportedLinearStagePosition)
Read 1 history items for RemoteEvent(ATSpectrograph, 0, settingsApplied)
Read 1 history items for RemoteEvent(ATSpectrograph, 0, settingsAppliedValues)
Read 4 history items for RemoteEvent(ATSpectro

[[None, None, None, None, None, None, None], [None, None, None, None]]

mountPositions DDS read queue is full (100 elements); data may be lost


In [6]:
# enable components if required
# Failed reason is missing configs - see below.
#await atcs.enable()
await atcs.enable({"atdome": "current", "ataos": "current", "athexapod": "current"})

Enabling all components
Gathering settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for ataos.
Complete settings for atpneumatics.
Complete settings for athexapod.
Complete settings for atdome.
Complete settings for atdometrajectory.
Settings versions: {'atmcs': '', 'atptg': '', 'ataos': '', 'atpneumatics': '', 'athexapod': '', 'atdome': '', 'atdometrajectory': ''}
[atmcs]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atptg]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[ataos]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atpneumatics]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
Unable to trans

RuntimeError: Failed to transition ['athexapod', 'atdome'] to <State.ENABLED: 2>.

In [7]:
await salobj.set_summary_state(atcs.rem.athexapod, salobj.State.ENABLED, settingsToApply='current')

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [8]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply='current')

[<State.FAULT: 3>, <State.STANDBY: 5>]

In [9]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply='current')

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [10]:
await latiss.enable()

Enabling all components
Gathering settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atcamera.
Complete settings for atspectrograph.
Complete settings for atheaderservice.
Complete settings for atarchiver.
Settings versions: {'atcamera': '', 'atspectrograph': '', 'atheaderservice': '', 'atarchiver': ''}
[atcamera]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
Unable to transition atspectrograph to <State.ENABLED: 2> NoneType: None
.
Traceback (most recent call last):
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/csc_utils.py", line 148, in set_summary_state
    await cmd.start(timeout=timeout)
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/topics/remote_command.py", line 446, in start
    return await cmd_info.next_ackcmd(timeout=timeout)
  File "/opt/lsst/src/ts_salobj/python/lsst/ts/salobj/topics/rem

RuntimeError: Failed to transition ['atspectrograph'] to <State.ENABLED: 2>.

In [11]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.STANDBY, settingsToApply='current')

[<State.FAULT: 3>, <State.STANDBY: 5>]

In [12]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.ENABLED, settingsToApply='current')

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [ ]:
# Everything now enabled 10:43 AM

In [13]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [14]:
# turn on ATAOS corrections just to make sure the mirror is under air
tmp = await atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=False)

In [15]:
# Run 1 bias as a test
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001


array([2021032300001])

In [ ]:
atcs.slew_icrs?

In [16]:
# Slew to starting position and take an image to check headers and tracking
await atcs.slew_icrs(ra=21.0, dec=-60.0, rot=-110.0, rot_type=RotType.PhysicalSky,
                      slew_timeout=240., stop_before_slew=False, wait_settle=True)

await asyncio.sleep(2)
# take a 60s dark
await latiss.take_darks(60.0, 1)


Setting rotator physical position to -110.0 deg. Rotator will track sky.
Parallactic angle: 3.3720203002251257 | Sky Angle: 13.061132570946796
Sending command
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -019.622 deg; delta Az= -178.548 deg; delta N1 = +000.000 deg; delta N2 = +000.575 deg 
[Telescope] delta Alt = -019.321 deg; delta Az= -179.870 deg; delta N1 = +000.000 deg; delta N2 = -000.003 deg 
[Telescope] delta Alt = -016.743 deg; delta Az= +174.208 deg; delta N1 = +000.000 deg; delta N2 = +000.001 deg 
[Telescope] delta Alt = -011.614 deg; delta Az= +168.220 deg; delta N1 = +000.000 deg; delta N2 = +000.001 deg 
[Telescope] delta Alt = -005.764 deg; delta Az= +162.233 deg; delta N1 = +000.000 deg; delta N2 = +000.002 deg 
[Telescope] delta Alt = -001.514 deg; delta Az= +156.246 deg; delta N1 = +000.000 deg; d

array([2021032300002])

In [18]:
# Now loop from dec of 60 to almost 90
# RA going from 19h to 23h
ra_start = 19.00
for dec in [-60.0, -65.0, -70.0, -75.0, -80.0, -82.0, -84.0, -86.0, -88.0, -89.0]:
    for ra_counter in range(17):
        ra = ra_start + 0.25 * ra_counter
        await atcs.slew_icrs(ra=ra, dec=dec, rot=-110.0, rot_type=RotType.PhysicalSky,
                              slew_timeout=240., stop_before_slew=False, wait_settle=True)

        await asyncio.sleep(2)
        # take a 60s dark
        await latiss.take_darks(60.0, 1)

Setting rotator physical position to -110.0 deg. Rotator will track sky.
Parallactic angle: 52.13072326924537 | Sky Angle: 69.16879143638283
Sending command
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got False
Telescope not in position
[Telescope] delta Alt = -007.280 deg; delta Az= +023.810 deg; delta N1 = -000.000 deg; delta N2 = +002.701 deg 
[Telescope] delta Alt = -006.108 deg; delta Az= +023.645 deg; delta N1 = -000.000 deg; delta N2 = +000.383 deg 
[Telescope] delta Alt = -000.859 deg; delta Az= +021.906 deg; delta N1 = -000.000 deg; delta N2 = +000.001 deg 
[Telescope] delta Alt = -000.002 deg; delta Az= +017.829 deg; delta N1 = -000.000 deg; delta N2 = +000.001 deg 
[Telescope] delta Alt = -000.002 deg; delta Az= +012.159 deg; delta N1 = +000.000 deg; delta N2 = +000.001 deg 
[Telescope] delta Alt = -000.001 deg; delta Az= +006.4

In [19]:
# For shutdown of system
await atcs.stop_tracking()

Stop tracking.
Tracking state: <AtMountState.TRACKINGENABLED: 9>
Tracking state: <AtMountState.STOPPING: 10>
In Position: True.


In [20]:
# turn off corrections
tmp = await atcs.rem.ataos.cmd_disableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [21]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [22]:
# Stopping here.  Tiago taking system.

In [ ]:
# Putting everything back in standby.
await atcs.shutdown()

In [ ]:
await atcs.rem.atdome.cmd_start.set_start(settingsToApply="test", timeout=30)

In [ ]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

In [ ]:
await latiss.standby()

In [ ]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atheaderservice, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.STANDBY)

In [ ]:
# Everything back in standby